# Watershed Delineation with pysheds
## Río Copiapó Basin · Chile

A complete watershed delineation workflow using free, open-access tools and data:

- **DEM**: [Copernicus GLO-30](https://registry.opendata.aws/copernicus-dem/) (30 m, free from AWS S3 — no credentials required)
- **Routing**: [pysheds](https://github.com/mdbartos/pysheds) (D8 flow direction)
- **Study area**: Río Copiapó basin, Atacama Region, Chile (~17,000 km²)

**Outputs:**
1. `results/watershed.shp` — watershed boundary polygon
2. `results/network.shp`   — stream network polylines
3. `results/watershed_map.png` — publication-ready map
4. Morphological parameters table (16 parameters)

---
> **Runtime note**: first run downloads ~400–600 MB of DEM tiles (cached after that).
> D8 pipeline on a ~17,000 km² basin takes ~5–10 min on a standard laptop.

## 1  Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LightSource
import rasterio

from utils import __version__
from utils.delineation import download_dem, delineate_watershed, snap_to_network
from utils.morphology import calculate_morphology

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
print(f"watershed-chile v{__version__} — Ready.")

## 2  Study Area and DEM Download

The Copiapó basin extends from the Pacific coast to the Atacama highlands (~4,000 m a.s.l.).
We define a bounding box that covers the full potential upstream area and download
the Copernicus GLO-30 DEM tiles automatically.

> To use a different basin, change `BBOX` and `OUTLET_LON / OUTLET_LAT` in cells 2 and 3.

In [ ]:
# Bounding box [xmin, ymin, xmax, ymax] in WGS84
# Covers the Copiapó basin from coast to Andes
BBOX = [-71.0, -28.5, -68.8, -26.2]

dem_path = download_dem(BBOX, cache_dir="../cache")
print(f"DEM ready: {dem_path}")

## 3  Outlet Point

We use a GeoJSON point located on the Río Copiapó just west of the city,
where the channel is clearly defined in the DEM (urban areas can distort D8 routing
in surface models like Copernicus GLO-30).

### Optional: pre-snap to BCN hydrographic network

If you have access to the official BCN river network
(available at [bcn.cl/siit/mapoteca](https://www.bcn.cl/siit/mapoteca)), you can snap
the outlet to the nearest authoritative channel line for improved accuracy.

**Steps:**
1. Download the "Red Hidrográfica" shapefile from the BCN Mapoteca.
2. Clip it to your bbox (the full national file is ~1 GB):
   ```python
   gdf = gpd.read_file("path/to/bcn_full.shp")
   gdf.cx[-71.0:-68.8, -28.5:-26.2].to_file("data/bcn_copiapo.shp")
   ```
3. Uncomment the `snap_to_network` call below.

In [ ]:
# Load outlet from GeoJSON
gdf_outlet = gpd.read_file("../data/copiapo_outlet.geojson")
OUTLET_LON = float(gdf_outlet.geometry.x.iloc[0])
OUTLET_LAT = float(gdf_outlet.geometry.y.iloc[0])

print(f"Outlet: {OUTLET_LON:.4f}°E  {OUTLET_LAT:.4f}°N")

# -- Optional BCN pre-snap (uncomment if you have the network shapefile) --
# OUTLET_LON, OUTLET_LAT = snap_to_network(
#     OUTLET_LON, OUTLET_LAT,
#     network_shp="data/bcn_copiapo.shp",
#     max_dist_m=500.0,
# )
# print(f"After BCN snap: {OUTLET_LON:.4f}°E  {OUTLET_LAT:.4f}°N")

## 4  D8 Hydrological Processing and Watershed Delineation

The D8 (deterministic eight-direction) algorithm routes flow to the steepest downslope
neighbour. The pipeline:

1. **fill_pits** — remove single-cell depressions that block drainage
2. **fill_depressions** — fill all sinks to their pour point elevation
3. **resolve_flats** — break ties on flat areas using a small gradient
4. **flowdir** — assign D8 direction to each cell
5. **accumulation** — count upstream cells draining through each cell
6. **snap** — move the outlet to the nearest high-accumulation cell
7. **catchment** — trace all cells that drain through the (snapped) outlet

In [ ]:
result = delineate_watershed(
    outlet_lon=OUTLET_LON,
    outlet_lat=OUTLET_LAT,
    dem_path=dem_path,
    out_dir="../results",
    acc_threshold=500,       # 500 cells × (30 m)² ≈ 0.45 km² contributing area
    snap_dist_m=1000.0,      # allow up to 1 km snap (useful near urban areas)
    network_threshold=5000,  # export only major channels (sparser map)
    crs_output="EPSG:32719", # UTM 19S — adjust to your UTM zone
    extract_network=True,
)

ws_area = result["watershed"].to_crs("EPSG:32719").geometry.area.iloc[0] / 1e6
print(f"\nWatershed area: {ws_area:,.0f} km²")
print(f"Saved: {result['watershed_shp']}")

## 5  Morphological Parameters

16 parameters computed from the watershed polygon and the DEM.

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Area | A_km2 | Basin area (km²) |
| Perimeter | P_km | Basin perimeter (km) |
| Gravelius | Kc | Compactness coefficient — 1 = circle, higher = more elongated |
| Form factor | Ff | A/L² — higher values indicate rounder, faster-responding basins |
| Circularity | Rc | 4πA/P² |
| Elongation | Re | 2√(A/π)/L |
| Relief | H | Hmax − Hmin (m) |
| Effective relief | Hm | Hmean − Hmin (m) |
| Channel slope | Scp | Main channel slope (m/m) |
| Basin slope | Scu | Mean slope across the basin (m/m) |
| Drainage density | Dd | Total channel length / area (km/km²) |

In [ ]:
morph = calculate_morphology(
    watershed_shp=result["watershed_shp"],
    dem_path=dem_path,
    acc_threshold=500,
    crs_work="EPSG:32719",
)

labels = {
    "A_km2":  "Area",            "P_km":   "Perimeter",
    "Kc":     "Gravelius (Kc)",  "Ff":     "Form factor (Ff)",
    "Rc":     "Circularity",     "Re":     "Elongation",
    "Hmin":   "Hmin (m a.s.l.)", "Hmax":   "Hmax (m a.s.l.)",
    "Hmed":   "Hmean (m a.s.l.)","H":      "Relief H (m)",
    "Hm":     "Eff. relief Hm",  "L_km":   "Channel length (km)",
    "Lcu_km": "Max flow path",   "Scp":    "Channel slope (m/m)",
    "Scu":    "Basin slope (m/m)","Dd":    "Drainage density",
}
units = {
    "A_km2": "km²", "P_km": "km", "Kc": "—", "Ff": "—", "Rc": "—", "Re": "—",
    "Hmin": "m", "Hmax": "m", "Hmed": "m", "H": "m", "Hm": "m",
    "L_km": "km", "Lcu_km": "km", "Scp": "m/m", "Scu": "m/m", "Dd": "km/km²",
}

df_morph = pd.DataFrame({
    "Parameter": [labels[k] for k in morph.index],
    "Value":     morph.values,
    "Units":     [units[k] for k in morph.index],
})

print(df_morph.to_string(index=False))

## 6  Publication-Ready Map

Hillshaded DEM background with watershed boundary and stream network overlay.

In [ ]:
with rasterio.open(dem_path) as src:
    dem_data = src.read(1).astype(float)
    nodata_val = src.nodata
    bounds = src.bounds

if nodata_val is not None:
    dem_data[dem_data == nodata_val] = np.nan

extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

ls = LightSource(azdeg=315, altdeg=35)
hs = ls.hillshade(np.nan_to_num(dem_data, nan=float(np.nanmin(dem_data))), vert_exag=2)

ws_wgs  = result["watershed"].to_crs("EPSG:4326")
net_wgs = result["network"].to_crs("EPSG:4326") if result["network"] is not None else None

fig, ax = plt.subplots(figsize=(11, 9), dpi=120)

ax.imshow(hs, cmap="gray", vmin=0, vmax=1,
          extent=extent, origin="upper", alpha=0.85, zorder=0)

ws_wgs.plot(ax=ax, facecolor=mcolors.to_rgba("steelblue", alpha=0.15),
            edgecolor="steelblue", linewidth=1.8, zorder=2, label="Watershed")

if net_wgs is not None:
    net_wgs.plot(ax=ax, color="#003580", linewidth=0.5, zorder=3, label="Stream network")

ax.plot(OUTLET_LON, OUTLET_LAT,
        marker="v", color="crimson", markersize=9,
        markeredgecolor="white", markeredgewidth=0.8,
        zorder=5, label="Outlet")

ws_b = ws_wgs.total_bounds
margin_x = (ws_b[2] - ws_b[0]) * 0.05
margin_y = (ws_b[3] - ws_b[1]) * 0.05
ax.set_xlim(ws_b[0] - margin_x, ws_b[2] + margin_x)
ax.set_ylim(ws_b[1] - margin_y, ws_b[3] + margin_y)

ax.set_xlabel("Longitude (°)", fontsize=11)
ax.set_ylabel("Latitude (°)", fontsize=11)
ax.set_title(
    f"Río Copiapó Watershed  ·  {morph['A_km2']:,.0f} km²  ·  Copernicus GLO-30 DEM",
    fontsize=12, pad=10,
)
ax.legend(loc="lower right", fontsize=10, framealpha=0.85)
ax.grid(True, linestyle=":", linewidth=0.4, alpha=0.6, color="gray")

plt.tight_layout()
out_png = "../results/watershed_map.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Map saved: {out_png}")

---

## Adapting to your own basin

1. Change `BBOX` and `OUTLET_LON / OUTLET_LAT` in cells 2 and 3.
2. Adjust `acc_threshold` to match your basin size (at 30 m, 500 cells ≈ 0.45 km²).
3. Set `crs_output` to the appropriate UTM zone for your area.
4. If the delineated area is unexpectedly small, the outlet may be in an urban section
   where the DSM blocks D8 routing — move it slightly upstream to a natural reach.
5. For better results in urban/vegetated areas, use NASADEM instead of Copernicus
   (requires a free [OpenTopography API key](https://opentopography.org/)).

## References

- Bartos, M. (2021). *pysheds: simple and fast watershed delineation in Python*. [github.com/mdbartos/pysheds](https://github.com/mdbartos/pysheds)
- Copernicus DEM GLO-30. (2022). *Copernicus Digital Elevation Model*. ESA / Airbus. [registry.opendata.aws/copernicus-dem](https://registry.opendata.aws/copernicus-dem/)